In [9]:
import joblib
import pandas as pd
import shap
import numpy as np
from sklearn.model_selection import train_test_split

In [10]:
log_reg_pipeline = joblib.load("../models/logreg_tuned_pipeline.pkl")
label_encoder = joblib.load("../models/label_encoder.pkl")
gene_names = joblib.load("../models/gene_names.pkl")

log_reg_tuned = log_reg_pipeline.named_steps["classifier"]

In [11]:
coefficients = log_reg_tuned.coef_
classes = log_reg_tuned.classes_

coef_df = pd.DataFrame(coefficients, index=classes, columns=gene_names)

top_n = 15
for cell_type in coef_df.index:
    top_genes = coef_df.loc[cell_type].sort_values(ascending=False).head(top_n)
    print(f"\nTop {top_n} genes pushing toward {cell_type}:")
    print(top_genes)


Top 15 genes pushing toward B cells:
CD79A        0.122746
MS4A1        0.109579
CD79B        0.104364
HLA-DQA1     0.083895
HLA-DRA      0.081150
LINC00926    0.080481
HLA-DQB1     0.079834
CD74         0.079129
TCL1A        0.072345
HLA-DPB1     0.069259
HLA-DPA1     0.068127
HLA-DRB1     0.067332
CD37         0.061810
HVCN1        0.057930
HLA-DMA      0.056493
Name: B cells, dtype: float32

Top 15 genes pushing toward CD14 Monocytes:
FTL       0.078547
LST1      0.070088
S100A9    0.069441
FCN1      0.067663
FTH1      0.067547
S100A8    0.067365
AIF1      0.066584
TYROBP    0.066043
FCER1G    0.061641
CDA       0.059262
LGALS1    0.056794
CST3      0.055956
CTSS      0.055423
SAT1      0.053707
NPC2      0.053296
Name: CD14 Monocytes, dtype: float32

Top 15 genes pushing toward CD4 T cells:
FYB             0.060697
MAL             0.059525
LTB             0.057846
TPT1            0.053359
IL32            0.047782
JUNB            0.046113
TNFRSF4         0.043566
AQP3            0.

In [12]:
rf_model = joblib.load("../models/random_forest.pkl")
xgb_model = joblib.load("../models/xgboost.pkl")
gene_names = joblib.load("../models/gene_names.pkl")

rf_importance = pd.Series(rf_model.feature_importances_, index=gene_names).sort_values(ascending=False)
xgb_importance = pd.Series(xgb_model.feature_importances_, index=gene_names).sort_values(ascending=False)

top_n = 20
print(f"Top {top_n} genes by Random Forest Generation:")
print(rf_importance.head(top_n))

print(f"\nTop {top_n} genes by XGBoost importance:")
print(xgb_importance.head(top_n))

Top 20 genes by Random Forest Generation:
NKG7        0.026512
HLA-DRB1    0.020115
FTL         0.019608
CD79A       0.018868
CD74        0.018123
HLA-DRA     0.015608
HLA-DPB1    0.014637
FTH1        0.014415
GPX1        0.014095
HLA-DPA1    0.013724
CCL5        0.013580
HLA-DQA1    0.013144
CD79B       0.012037
GZMA        0.010865
TYROBP      0.010816
CST7        0.010638
OAZ1        0.010581
CTSW        0.010567
MALAT1      0.010423
HLA-DQB1    0.010364
dtype: float64

Top 20 genes by XGBoost importance:
CD79A       0.060594
FTL         0.050007
HLA-DRA     0.029400
SERPINF1    0.028761
NKG7        0.028731
HLA-DRB1    0.026615
CST3        0.025130
CLEC10A     0.024682
CST7        0.018312
KIAA0101    0.018295
FERMT3      0.016922
SMC2        0.015387
TYROBP      0.015025
ZWINT       0.014218
ENHO        0.013881
STMN1       0.013547
CKS1B       0.013050
FCER1A      0.012788
CD74        0.012030
FGFBP2      0.009142
dtype: float32


In [13]:
rf_model = joblib.load("../models/random_forest.pkl")
xgb_model = joblib.load("../models/xgboost.pkl")
label_encoder = joblib.load("../models/label_encoder.pkl")
gene_names = joblib.load("../models/gene_names.pkl")

X_train_unscaled = np.load("../models/X_train_unscaled.npy")
y_train_encoded = np.load("../models/y_train_encoded.npy")

# Stratified subsample so rare classes aren't dropped entirely (Platelets: 9, Dendritic cells: 34)
_, X_sample, _, y_sample = train_test_split(
    X_train_unscaled, y_train_encoded,
    test_size=300,
    stratify=y_train_encoded,
    random_state=42
)

# data= is required: without a background dataset, TreeExplainer falls back to
# feature_perturbation="tree_path_dependent" and computes expected_value from the
# model's internal (class_weight="balanced"-reweighted) root-node values, which are
# exactly 1/6 for every class regardless of true frequency. model_output="probability"
# is a no-op for RandomForestClassifier specifically (its native tree_output is
# already "probability"), but is kept for clarity/portability.
#
# Passing X_sample directly as data= wraps it in shap.maskers.Independent with its
# DEFAULT max_samples=100, which silently re-subsamples our stratified 300 down to a
# non-stratified 100 (verified: this drops Platelets to 0/100 and Dendritic cells to
# 2/100 in the background, defeating the entire point of stratifying). Building the
# masker explicitly with max_samples=300 keeps all 300 stratified cells, including
# the 1 Platelet and 5 Dendritic cells, in the background.
rf_masker = shap.maskers.Independent(X_sample, max_samples=300)
rf_explainer = shap.TreeExplainer(rf_model, data=rf_masker, model_output="probability")
rf_shap_values = rf_explainer.shap_values(X_sample)

print(type(rf_shap_values))
if isinstance(rf_shap_values, list):
    print(f"List of {len(rf_shap_values)} arrays, each shaped {rf_shap_values[0].shape}")
else:
    print(f"Single array shaped {rf_shap_values.shape}")

100%|===================| 1795/1800 [01:54<00:00]        

<class 'numpy.ndarray'>
Single array shaped (300, 2000, 6)


In [14]:
# Averaging signed SHAP values across ALL 300 samples (mixed classes) collapses to
# ~0 noise, because those same 300 samples are also the explainer's background —
# the mean deviation from the background's own mean is zero by construction,
# regardless of biology. Restricting to samples truly labeled as class K avoids this
# (that subset is not representative of the background as a whole), and keeps the
# same signed "genes pushing toward X" framing as the coefficient results.

sample_labels = label_encoder.inverse_transform(y_sample)

top_n = 15
for i, cell_type in enumerate(rf_model.classes_):
    mask = sample_labels == cell_type
    n = mask.sum()
    class_shap = rf_shap_values[mask, :, i]
    mean_shap = class_shap.mean(axis=0)
    top_genes = pd.Series(mean_shap, index=gene_names).sort_values(ascending=False).head(top_n)
    print(f"\nTop {top_n} genes for {cell_type} (Random Forest, n={n} true-label samples):")
    print(top_genes)


Top 15 genes for B cells (Random Forest, n=39 true-label samples):
CD74        0.076653
HLA-DRA     0.062351
CD79A       0.060990
HLA-DRB1    0.053208
CD79B       0.038960
HLA-DPB1    0.036169
HLA-DPA1    0.028478
HLA-DQA1    0.026539
MS4A1       0.024393
HLA-DQB1    0.024373
S100A4      0.017340
CD37        0.016225
IL32        0.013327
TCL1A       0.013209
LGALS1      0.011667
dtype: float64

Top 15 genes for CD14 Monocytes (Random Forest, n=73 true-label samples):
FTL       0.066175
FTH1      0.045783
TYROBP    0.040557
CST3      0.038505
LYZ       0.028256
S100A9    0.024336
LST1      0.024255
AIF1      0.023871
S100A8    0.022518
LGALS1    0.021385
FCER1G    0.019744
CTSS      0.017838
FCN1      0.016240
LGALS2    0.014370
OAZ1      0.012850
dtype: float64

Top 15 genes for CD4 T cells (Random Forest, n=134 true-label samples):
NKG7        0.027468
HLA-DRB1    0.022210
HLA-DRA     0.022180
CD74        0.021199
HLA-DPB1    0.014844
FTL         0.013822
TYROBP      0.013165
HLA-DPA

In [15]:
print("Expected value by rf_model.classes_ order:")
for cls, val in zip(rf_model.classes_, rf_explainer.expected_value):
    print(f"  {cls}: {val:.4f}")

sample_labels = label_encoder.inverse_transform(y_sample)
print("\nTrue class frequencies in this 300-cell sample:")
print(pd.Series(sample_labels).value_counts(normalize=True).round(4))

Expected value by rf_model.classes_ order:
  B cells: 0.1411
  CD14 Monocytes: 0.2382
  CD4 T cells: 0.4072
  Dendritic cells: 0.0242
  NK cells: 0.1856
  Platelets: 0.0037

True class frequencies in this 300-cell sample:
CD4 T cells        0.4467
CD14 Monocytes     0.2433
NK cells           0.1600
B cells            0.1300
Dendritic cells    0.0167
Platelets          0.0033
Name: proportion, dtype: float64


In [16]:
xgb_masker = shap.maskers.Independent(X_sample, max_samples=300)
xgb_explainer = shap.TreeExplainer(xgb_model, data=xgb_masker, model_output="probability")
xgb_shap_values = xgb_explainer.shap_values(X_sample)

print(type(xgb_shap_values))
if isinstance(xgb_shap_values, list):
    print(f"List of {len(xgb_shap_values)} arrays, each shaped {xgb_shap_values[0].shape}")
else:
    print(f"Single array shaped {xgb_shap_values.shape}")

xgb_class_names = label_encoder.inverse_transform(xgb_model.classes_)
print("\nExpected value by class:")
for cls, val in zip(xgb_class_names, xgb_explainer.expected_value):
    print(f"    {cls}:  {val:.4f}")

sample_labels = label_encoder.inverse_transform(y_sample)
print("\nTrue class frequencies in this 300-cell sample:")
print(pd.Series(sample_labels).value_counts(normalize=True).round(4))

Exception: Model does not have a known objective or output type! When model_output is not "raw" then we need to know the model's objective or link function.

In [ ]:
print("xgb_model.objective:", xgb_model.objective)
print("get_xgb_params() objective:", xgb_model.get_xgb_params().get("objective"))

booster = xgb_model.get_booster()
import json
config = json.loads(booster.save_config())
print("Booster's own config objective:", config["learner"]["objective"])